# RAG Ecosystem

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/release/python-3100/) [![LangChain](https://img.shields.io/badge/LangChain-%23007ACC.svg?logo=LangChain)](https://www.langchain.com/) [![DeepEval](https://img.shields.io/badge/DeepEval-Evaluation-orange)](https://github.com/confident-ai/deepeval) [![RAGAS](https://img.shields.io/badge/RAGAS-Evaluation-blueviolet)](https://github.com/explodinggradients/ragas) [![OpenAI](https://img.shields.io/badge/OpenAI-API-lightgrey)](https://openai.com/) [![Cohere](https://img.shields.io/badge/Cohere-API-yellowgreen)](https://cohere.com/) [![Medium](https://img.shields.io/badge/Medium-Blog-black?logo=medium)](https://medium.com/@fareedkhandev/8f23349b96a4)

Hello 👋🏻 I'm Yashh and in this notebook I've built and implemented a complete RAG Based AI System with optimizing each component. 

 I've made this notebook in such a way that it becomes easy for me to revise and resuse my work, and also for the easy understanding for the viewers.

![image.png](attachment:image.png)

> The Advanced components that we'll be working on in detail : 

- **Query Transformations:** Rewriting user questions to be more effective for retrieval.
- **Intelligent Routing:** Directing a query to the correct data source or a specialized tool.
- **Indexing:** Creating a multi-layered knowledge base.
- **Retrieval and Re-ranking:** Filtering noise and prioritizing the most relevant context.
- **Self-Correcting Agentic Flows:** Building systems that can grade and improve their own work.
- **End-to-End Evaluation:** Objectively measuring the performance of the entire pipeline.

and much more …

> Let's create the RAG!

## Table of Contents

- [Understanding Basic RAG System](#part1)
  - [Indexing Phase](#part1-1)
  - [Retrieval](#part1-2)
  - [Generation](#part1-3)
- [Advanced Query Transformations](#part2)
  - [Multi-Query Generation](#part2-1)
  - [RAG-Fusion](#part2-2)
  - [Decomposition](#part2-3)
  - [Step-Back Prompting](#part2-4)
  - [HyDE](#part2-5)
- [Routing & Query Construction](#part3)
  - [Logical Routing](#part3-1)
  - [Semantic Routing](#part3-2)
  - [Query Structuring](#part3-3)
- [Advanced Indexing Strategies](#part4)
  - [Multi-Representation Indexing](#part4-1)
  - [Hierarchical Indexing (RAPTOR) Knowledge Tree](#part4-2)
  - [Token-Level Precision (ColBERT)](#part4-3)
- [Advanced Retrieval & Generation](#part5)
  - [Dedicated Re-ranking](#part5-1)
  - [Self-Correction using AI Agents](#part5-2)
  - [Impact of Long Context](#part5-3)
- [Manual RAG Evaluation](#part6)
  - [The Core Metrics: What Should We Measure?](#part6-1)
  - [Building Evaluators from Scratch with LangChain](#part6-2)
- [Evaluation with Frameworks](#part7)
  - [Rapid Evaluation with deepeval](#part7-1)
  - [Another Powerful Alternative with grouse](#part7-2)
  - [Evaluation with RAGAS](#part7-3)
- [Summarizing our Work](#part8)

> small caveat to know, we'll be using python 3.10 for this project specifically. I have noticed unusual errors because my system is running on python 3.13. 

> Python 3.10 is stable and supports wider tools that we'll be using for this project

> if your system is running on 3.11+ too, then we explicitly install 3.10 for this project and here are the 3 commands to run. 

1. rm -rf venv (if already made)
2. python3.10 -m venv venv
3. source venv/bin/activate (for mac)
4. source venv/Scripts/activate (for windows)

<a id='part1'></a>
# Basics of a RAG System

Before we get into anything, let’s install core Python libraries commonly used for AI products, such as LangChain and others.

In [1]:
!pip install langchain langchain_community langchain-openai langchainhub chromadb tiktoken

We now set the environment variables for tracing and other tasks, such as the LLMs API provider we will be using.

In [2]:
import os
from dotenv import load_dotenv

# Load variables from .env file
load_dotenv()

os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

You can obtain your `LangSmith` API key from [their official documentation](https://www.langchain.com/langsmith) to trace our RAG product throughout this blog. For the LLM, we will be using the `Groq` API but as you may already know, `LangChain` supports a variety of LLM providers as well.

Although i'm not using it, `LANGCHAIN_PROJECT` is optional but strongly recommended—it organizes traces in LangSmith.

The core RAG pipeline is the foundation of any advanced system, and understanding its components is important. Therefore, before going into the details of advanced components, we first need to understand the core logic of how a RAG system works, **but you can skip this section if you are already aware of how RAG system works.**


This simplest RAG can be break into three components:

- **Indexing**: Organize and store data in a structured format to enable efficient searching.
- **Retrieval**: Search and fetch relevant data based on a query or input.
- **Generation**: Create a final response or output using the retrieved data.

Let’s build this simple pipeline from the ground up to see how each piece works.

<a id='part1-1'></a>
## Indexing Phase

Before our RAG system can answer any questions, it needs knowledge to draw from. For this, we’ll use a `WebBaseLoader` to pull content directly from [Lilian Weng's excellent blog post](https://lilianweng.github.io/posts/2023-06-23-agent/) on LLM-powered agents.


In [3]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Initialize a web document loader with specific parsing instructions
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),  # URL of the blog post to load
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")  # Only parse specified HTML classes
        )
    ),
)

# Load the filtered content from the web page into documents
docs = loader.load()

/Users/yashhmahajan/Desktop/Comprehensive RAG Ecosystem/rageco/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


The `bs_kwargs` argument helps us target only the relevant HTML tags (`post-content`, `post-title`, etc.), cleaning up our data from the start.

Now that we have the document, we face our first challenge. Feeding a massive document directly into an LLM is inefficient and often impossible due to context window limits.

> This is why **chunking** is a critical step. We need to break the document into smaller, semantically meaningful pieces.

The `RecursiveCharacterTextSplitter` is the recommended tool for this job because it intelligently tries to keep paragraphs and sentences intact.

In [4]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

# Create a text splitter to divide text into chunks of 1000 characters with 200-character overlap
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Split the loaded documents into smaller chunks
splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} document chunks")

Created 63 document chunks


With `chunk_size=1000`, we are creating chunks of 1000 characters, and `chunk_overlap=200` ensures there is some continuity between them, which helps preserve context.

Our text is now split, but it’s still just text. To perform similarity searches, we need to convert these chunks into numerical representations called **embeddings**. We will then store these embeddings in a **vector store**, which is a specialized database designed for efficient searching of vectors.

* The `Chroma` vector store and `HuggingFaceEmbeddings` make this incredibly simple. 

Just like an LLM API Key, we also need to get API key for embeddings and we have muliple embedding models to choose from. 

If you plan on using this project for commerical usage, you might want to use paid embedding models from OpenAI or others.

But for now, we'll be using a free & open source model.

* Major advantage is huggingfaceembeddings are open source and can be used without any API keys. Also it is free! 

The following line handles both embedding and indexing in one go.

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Prepend retrieval instruction to each document (BGE optimization)
splits = [
    Document(
        page_content="Represent this document for retrieval: " + doc.page_content,
        metadata=doc.metadata
    )
    for doc in splits
]

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "mps"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 4
    }
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding
)

/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/3145315280.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5731.26it/s]


I have a Macbook M2 8GB (as i make this project) and if you beleive your pc has more ram and cpu power, you can go for bigger and better models like `BAAI/bge-large-en-v1.5` or `BAAI/bge-m3`

* `BAAI/bge-small-en-v1.5` - uses 200MB of VRAM
* `BAAI/bge-base-en-v1.5` - uses 600MB of VRAM
* `BAAI/bge-large-en-v1.5` - uses 1.8GB of VRAM
* `BAAI/bge-m3` - uses 2.2BG of VRAM (only choose when working with multilingual content)

### With our knowledge indexed, we are now ready to start asking questions.

<a id='part1-2'></a>
## Retrieval

The vector store is our library, and the **retriever** is our smart librarian. It takes a user’s query, embeds it, and then fetches the most semantically similar chunks from the vector store.


First, we create a retreiver from a vector store

In [6]:
retriever = vectorstore.as_retriever()

Let’s test it. We’ll ask a question and see what our retriever finds.

In [7]:
# Retrieve relevant documents for a query
docs = retriever.invoke("What is Task Decomposition?")

# Print the content of the first retrieved document
print(docs[0].page_content)

Represent this document for retrieval: Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation of the model’s thinking process.
Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS (depth-first search) with each state evaluated by a classifier (via a prompt) or majority vote.


As you can see, the retriever successfully pulled the most relevant chunk from the blog post that directly discusses “Task decomposition.” This piece of context is exactly what the LLM needs to form an accurate answer.

<a id='part1-3'></a>
## Generation

We have our context, but we need an LLM to read it and formulate a human-friendly answer. This is the **“Generation”** step in RAG.


We'll be using LangChain Hub for the templates.

> For those who dont know, the Hub is a centralized platform for sharing, managing, and collaborating on LangChain artifacts, most importantly for now, prompt templates.

> It is integrated directly into LangSmith and serves as a "GitHub for prompts

you can find it here - https://smith.langchain.com/hub?organizationId=63336e7b-fed5-451d-acf3-8ac3fe222056

First, we need a good prompt template. This instructs the LLM on how to behave. Instead of writing our own, we can pull a pre-optimized one from LangChain Hub.

In [8]:
from langchainhub import Client
client = Client()
prompt = client.pull("rlm/rag-prompt")
print(prompt)

/var/folders/t1/wt965jp13hb3gls9w5dcy1bm0000gn/T/ipykernel_39802/2354009971.py:3: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt = client.pull("rlm/rag-prompt")
/Users/yashhmahajan/Desktop/Comprehensive RAG Ecosystem/rageco/lib/python3.10/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)


{"id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"messages": [{"id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"prompt": {"id": ["langchain", "prompts", "prompt", "PromptTemplate"], "lc": 1, "type": "constructor", "kwargs": {"template": "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:", "input_variables": ["question", "context"], "template_format": "f-string"}}}}], "input_variables": ["question", "context"]}}


Next, we initialize our LLM

In [9]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

Now for the final step: chaining everything together. Using the LangChain Expression Language (LCEL), we can pipe the output of one component into the input of the next.

In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

retriever = vectorstore.as_retriever()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

format_docs_runnable = RunnableLambda(format_docs)

prompt = ChatPromptTemplate.from_template(
    "Answer the question based only on the context below:\n\n{context}\n\nQuestion: {question}"
)

rag_chain = (
    {
        "context": retriever | format_docs_runnable,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

Let’s break down this chain:

1. `{"context": retriever | format_docs_runnable, "question": RunnablePassthrough()}`: This part runs in parallel. It sends the user's question to the `retriever` to get documents, which are then formatted into a single string by `format_docs`. Simultaneously, `RunnablePassthrough` passes the original question through unchanged.
2. `| prompt`: The context and question are fed into our prompt template.
3. `| llm`: The formatted prompt is sent to the LLM.
4. `| StrOutputParser()`: This cleans up the LLM's output into a simple string.

Now, let’s invoke the entire chain.

In [11]:
response = rag_chain.invoke("What is Task Decomposition?")
print(response)

Task decomposition is the process of breaking down a complicated task into smaller and simpler steps. This can be done in several ways, including: 

1. Using large language models (LLMs) with simple prompting, 
2. Using task-specific instructions, 
3. With human inputs, 
4. Utilizing techniques like Chain of Thought (CoT) which instructs the model to "think step by step", 
5. Tree of Thoughts which explores multiple reasoning possibilities at each step, and 
6. LLM+P, which involves relying on an external classical planner to do long-horizon planning using the Planning Domain Definition Language (PDDL). 

The goal of task decomposition is to transform big tasks into multiple manageable tasks, making it easier to understand and complete the task.


And there we have it, our RAG pipeline successfully retrieved relevant information about **“Task Decomposition”** and used it to generate a concise, accurate answer. This simple chain forms the foundation upon which we will build more advanced and powerful capabilities.